# BÀI THỰC HÀNH CHƯƠNG 2 (PHẦN 2): CANNY EDGE DETECTOR

**Mục tiêu:**
- Hiểu rõ nguyên lý hoạt động của thuật toán Canny Edge Detector.
- Thực hành sử dụng Canny với các thư viện OpenCV và Scikit-image.
- Phân tích ảnh hưởng của các tham số và so sánh với các phương pháp phát hiện cạnh khác.
- Kết hợp Canny với các kỹ thuật xử lý ảnh nâng cao.

---

# I. LÝ THUYẾT

## 1.1. Tìm hiểu về các bước của thuật toán Canny

### a) Giải thích chi tiết từng bước

Thuật toán Canny Edge Detector (do John F. Canny đề xuất năm 1986) là một thuật toán phát hiện cạnh đa bước, được coi là **chuẩn vàng (gold standard)** trong lĩnh vực phát hiện cạnh. Thuật toán gồm **5 bước chính**:

---

**Bước 1: Giảm nhiễu bằng bộ lọc Gaussian (Gaussian Smoothing)**

- **Mục đích:** Loại bỏ nhiễu trong ảnh vì nhiễu có thể tạo ra các cạnh giả (false edges).
- **Phương pháp:** Áp dụng bộ lọc Gaussian (convolution) lên ảnh gốc.
- **Công thức kernel Gaussian 2D:**

$$G(x, y) = \frac{1}{2\pi\sigma^2} e^{-\frac{x^2 + y^2}{2\sigma^2}}$$

- Trong đó $\sigma$ (sigma) là độ lệch chuẩn, quyết định mức độ làm mịn.
- Kích thước kernel thường là $5 \times 5$.
- $\sigma$ càng lớn → ảnh càng mịn → mất nhiều chi tiết nhỏ nhưng loại bỏ nhiễu tốt hơn.

---

**Bước 2: Tính toán Gradient (Gradient Computation)**

- **Mục đích:** Xác định cường độ và hướng của gradient tại mỗi pixel.
- **Phương pháp:** Sử dụng toán tử Sobel theo 2 hướng $x$ và $y$:

$$G_x = \begin{bmatrix} -1 & 0 & +1 \\ -2 & 0 & +2 \\ -1 & 0 & +1 \end{bmatrix} * I, \quad G_y = \begin{bmatrix} -1 & -2 & -1 \\ 0 & 0 & 0 \\ +1 & +2 & +1 \end{bmatrix} * I$$

- **Cường độ gradient (magnitude):**

$$|G| = \sqrt{G_x^2 + G_y^2}$$

- **Hướng gradient (direction):**

$$\theta = \arctan\left(\frac{G_y}{G_x}\right)$$

- Hướng gradient được làm tròn về 4 hướng chính: **0°, 45°, 90°, 135°**.

---

**Bước 3: Triệt tiêu phi cực đại (Non-Maximum Suppression - NMS)**

- **Mục đích:** Làm mỏng các cạnh, chỉ giữ lại các pixel có cường độ gradient cực đại cục bộ.
- **Phương pháp:**
  - Tại mỗi pixel, so sánh cường độ gradient của nó với 2 pixel lân cận **theo hướng gradient**.
  - Nếu pixel hiện tại có cường độ lớn nhất → **giữ lại**.
  - Nếu không → **loại bỏ (đặt bằng 0)**.
- **Kết quả:** Các cạnh trở nên mỏng (thin edges), chỉ còn 1 pixel chiều rộng.

---

**Bước 4: Ngưỡng kép (Double Thresholding / Hysteresis Thresholding)**

- **Mục đích:** Phân loại các pixel cạnh thành 3 nhóm dựa trên 2 ngưỡng.
- **Hai ngưỡng:** Ngưỡng cao ($T_{high}$) và ngưỡng thấp ($T_{low}$).
- **Phân loại:**
  - $|G| \geq T_{high}$: **Cạnh mạnh (Strong edge)** → chắc chắn là cạnh.
  - $T_{low} \leq |G| < T_{high}$: **Cạnh yếu (Weak edge)** → có thể là cạnh.
  - $|G| < T_{low}$: **Không phải cạnh** → loại bỏ.
- **Tỷ lệ khuyến nghị:** $T_{high} : T_{low} = 2:1$ hoặc $3:1$.

---

**Bước 5: Theo dõi cạnh bằng trễ (Edge Tracking by Hysteresis)**

- **Mục đích:** Quyết định các cạnh yếu có phải là cạnh thật hay không.
- **Phương pháp:**
  - Nếu một cạnh yếu **kết nối** (liền kề) với ít nhất một cạnh mạnh → **giữ lại** (coi là cạnh thật).
  - Nếu một cạnh yếu **không kết nối** với cạnh mạnh nào → **loại bỏ** (coi là nhiễu).
  - Sử dụng thuật toán duyệt đồ thị (BFS/DFS) trên vùng lân cận $8$ pixel.
- **Kết quả cuối cùng:** Bản đồ cạnh nhị phân (binary edge map) chất lượng cao.

### b) So sánh Canny với các thuật toán Sobel và Laplacian

| Tiêu chí | **Sobel** | **Laplacian** | **Canny** |
|:---|:---|:---|:---|
| **Nguyên lý** | Đạo hàm bậc 1 (gradient) | Đạo hàm bậc 2 (Laplacian) | Đa bước: Gaussian + Gradient + NMS + Hysteresis |
| **Loại cạnh** | Cạnh dày (thick) | Cạnh kép (double edges) tại zero-crossing | Cạnh mỏng, sắc nét (1 pixel) |
| **Xử lý nhiễu** | Nhạy với nhiễu vừa phải | **Rất nhạy** với nhiễu | **Tốt nhất** nhờ Gaussian smoothing |
| **Kết quả** | Gradient magnitude map | Zero-crossing map | Binary edge map chính xác |
| **Tham số** | Kích thước kernel | Kích thước kernel | $\sigma$, $T_{low}$, $T_{high}$ |
| **Độ phức tạp** | Thấp | Thấp | **Cao hơn** (nhiều bước xử lý) |
| **Chất lượng** | Trung bình | Thấp (do nhạy nhiễu) | **Cao nhất** |
| **Ứng dụng** | Tiền xử lý, feature extraction | Phát hiện blob, sharpening | Phát hiện cạnh chính xác, ứng dụng thị giác máy |

## 1.2. Các tham số của thuật toán và ảnh hưởng của chúng

### a) Sigma ($\sigma$) trong Gaussian Filter

- **$\sigma$ nhỏ (ví dụ: 0.5 - 1.0):**
  - Làm mịn ít → giữ lại nhiều chi tiết và cạnh nhỏ.
  - Nhưng cũng giữ lại nhiều nhiễu → phát hiện nhiều cạnh giả.
  
- **$\sigma$ lớn (ví dụ: 2.0 - 5.0):**
  - Làm mịn mạnh → loại bỏ nhiễu hiệu quả.
  - Nhưng cũng làm mờ các cạnh yếu và chi tiết nhỏ → bỏ sót cạnh.
  
- **Kết luận:** Cần chọn $\sigma$ phù hợp để cân bằng giữa loại bỏ nhiễu và giữ chi tiết. Giá trị $\sigma = 1.0 - 1.4$ thường cho kết quả tốt trong thực tế.

### b) Ngưỡng thấp ($T_{low}$) và ngưỡng cao ($T_{high}$)

- **Ngưỡng cao ($T_{high}$):**
  - $T_{high}$ **cao**: Chỉ phát hiện các cạnh mạnh → ít cạnh nhưng chính xác, có thể bỏ sót cạnh quan trọng.
  - $T_{high}$ **thấp**: Phát hiện nhiều cạnh hơn → nhưng có thể bao gồm cạnh giả.
  
- **Ngưỡng thấp ($T_{low}$):**
  - $T_{low}$ **cao**: Loại bỏ nhiều cạnh yếu → cạnh bị đứt đoạn.
  - $T_{low}$ **thấp**: Giữ lại nhiều cạnh yếu → cạnh liền mạch hơn nhưng nhiều nhiễu hơn.
  
- **Mối quan hệ giữa hai ngưỡng:**
  - Tỷ lệ $T_{high} / T_{low}$ thường từ **2:1** đến **3:1**.
  - Ví dụ: $T_{low} = 50$, $T_{high} = 150$ (tỷ lệ 3:1).
  - Tỷ lệ quá lớn → nhiều cạnh bị loại bỏ; Tỷ lệ quá nhỏ → nhiều nhiễu.

## 1.3. Ưu điểm, nhược điểm và ứng dụng thực tế

### a) So sánh với các thuật toán khác

| Tiêu chí | **Canny** | **Sobel** | **Laplacian** | **Prewitt** |
|:---|:---|:---|:---|:---|
| **Độ chính xác** | ⭐⭐⭐⭐⭐ Cao nhất | ⭐⭐⭐ Trung bình | ⭐⭐ Thấp | ⭐⭐⭐ Trung bình |
| **Tốc độ** | ⭐⭐⭐ Chậm hơn | ⭐⭐⭐⭐⭐ Nhanh | ⭐⭐⭐⭐⭐ Nhanh | ⭐⭐⭐⭐⭐ Nhanh |
| **Khả năng xử lý nhiễu** | ⭐⭐⭐⭐⭐ Tốt nhất | ⭐⭐⭐ Trung bình | ⭐ Kém | ⭐⭐⭐ Trung bình |
| **Cạnh mỏng** | ✅ Có (nhờ NMS) | ❌ Không | ❌ Không | ❌ Không |
| **Cạnh liên tục** | ✅ Có (nhờ Hysteresis) | ❌ Không | ❌ Không | ❌ Không |

### b) Lĩnh vực sử dụng phổ biến

- **Thị giác máy tính (Computer Vision):** Phát hiện đối tượng, nhận dạng khuôn mặt, ADAS (hỗ trợ lái xe).
- **Y tế (Medical Imaging):** Phát hiện biên của khối u, phân đoạn cơ quan trong ảnh MRI/CT.
- **Robot và tự động hóa:** Điều hướng robot, kiểm tra chất lượng sản phẩm.
- **Xử lý tài liệu:** OCR (nhận dạng ký tự), phát hiện vùng văn bản.
- **Viễn thám (Remote Sensing):** Phân tích ảnh vệ tinh, phát hiện ranh giới địa lý.

### c) Ví dụ cụ thể về ứng dụng

1. **Xe tự lái (Self-driving cars):** Canny được sử dụng để phát hiện làn đường (lane detection) trên đường, kết hợp với biến đổi Hough để xác định đường thẳng.
2. **Chẩn đoán y khoa:** Phát hiện đường viền của khối u trong ảnh X-quang hoặc MRI, hỗ trợ bác sĩ chẩn đoán.
3. **Kiểm tra chất lượng công nghiệp:** Phát hiện vết nứt, khuyết tật trên bề mặt sản phẩm (ví dụ: chip điện tử, linh kiện ô tô).
4. **Nhận dạng biển số xe:** Canny giúp tách biên các ký tự trên biển số, phục vụ cho hệ thống đọc biển số tự động (ANPR).

**Ưu điểm chính của Canny:**
- Phát hiện cạnh chính xác, mỏng, liên tục.
- Khả năng lọc nhiễu tốt nhờ bước Gaussian.
- Có thể điều chỉnh linh hoạt qua các tham số.

**Nhược điểm:**
- Tốc độ xử lý chậm hơn so với Sobel/Laplacian.
- Cần phải tinh chỉnh tham số ($\sigma$, $T_{low}$, $T_{high}$) cho từng loại ảnh.
- Không hoạt động tốt với ảnh có gradient thay đổi chậm (soft edges).
- Kết quả phụ thuộc nhiều vào chất lượng ảnh đầu vào.

---
# II. BÀI TẬP THỰC HÀNH

## 2.0. Cài đặt thư viện và import

In [ ]:
# ============================================================
# Import các thư viện cần thiết
# ============================================================
import cv2
import numpy as np
import matplotlib.pyplot as plt
from skimage import data, feature, filters, color, util, io
from skimage.feature import canny as skimage_canny
from scipy import ndimage

# Cấu hình matplotlib hiển thị trong notebook
%matplotlib inline
plt.rcParams['figure.figsize'] = (14, 8)
plt.rcParams['font.size'] = 12
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['figure.dpi'] = 100

print("OpenCV version:", cv2.__version__)
print("Các thư viện đã được import thành công!")

In [ ]:
# ============================================================
# Hàm tiện ích dùng chung cho toàn bộ notebook
# ============================================================

def show_images(images, titles, rows=1, figsize=(16, 6), cmap_list=None):
    """
    Hiển thị nhiều ảnh trên cùng một figure.
    
    Parameters:
        images: list các ảnh cần hiển thị
        titles: list tiêu đề cho mỗi ảnh
        rows: số hàng hiển thị
        figsize: kích thước figure
        cmap_list: list colormap cho mỗi ảnh (None = tự động)
    """
    cols = len(images) // rows + (1 if len(images) % rows != 0 else 0)
    fig, axes = plt.subplots(rows, cols, figsize=figsize)
    if rows == 1 and cols == 1:
        axes = [axes]
    elif rows == 1 or cols == 1:
        axes = axes.flatten()
    else:
        axes = axes.flatten()
    
    for i, (img, title) in enumerate(zip(images, titles)):
        ax = axes[i]
        if cmap_list and cmap_list[i]:
            ax.imshow(img, cmap=cmap_list[i])
        elif len(img.shape) == 2:
            ax.imshow(img, cmap='gray')
        else:
            ax.imshow(img)
        ax.set_title(title, fontweight='bold')
        ax.axis('off')
    
    # Ẩn các axes thừa
    for j in range(len(images), len(axes)):
        axes[j].axis('off')
    
    plt.tight_layout()
    plt.show()


def load_sample_image():
    """
    Tải ảnh mẫu từ skimage.data.
    Trả về ảnh gốc (RGB) và ảnh xám (grayscale).
    """
    # Sử dụng ảnh camera (grayscale) có sẵn trong skimage
    img_gray = data.camera()  # Ảnh xám 512x512
    # Tạo phiên bản RGB để hiển thị
    img_rgb = np.stack([img_gray]*3, axis=-1)
    return img_rgb, img_gray


# Tải ảnh mẫu
img_rgb, img_gray = load_sample_image()

# Hiển thị ảnh gốc
show_images(
    [img_rgb, img_gray],
    ['Ảnh gốc (RGB)', 'Ảnh xám (Grayscale)'],
    figsize=(12, 5)
)
print(f"Kích thước ảnh: {img_gray.shape}")
print(f"Kiểu dữ liệu: {img_gray.dtype}")
print(f"Giá trị min/max: {img_gray.min()} / {img_gray.max()}")

## 2.1. Thực hiện thuật toán Canny bằng các thư viện

### a) Canny với OpenCV (`cv2.Canny`)

In [ ]:
# ============================================================
# 2.1a - Canny Edge Detection với OpenCV
# ============================================================

# Áp dụng Gaussian Blur trước để giảm nhiễu
img_blur = cv2.GaussianBlur(img_gray, (5, 5), sigmaX=1.4)

# Áp dụng Canny với OpenCV
# Tham số: ảnh đầu vào, ngưỡng thấp, ngưỡng cao
edges_cv2 = cv2.Canny(img_blur, threshold1=50, threshold2=150)

# Hiển thị kết quả
show_images(
    [img_gray, img_blur, edges_cv2],
    ['Ảnh gốc (Grayscale)',
     'Sau Gaussian Blur (σ=1.4)',
     'Canny OpenCV\n(T_low=50, T_high=150)'],
    figsize=(16, 5)
)

print("=" * 60)
print("KẾT QUẢ CANNY VỚI OPENCV")
print("=" * 60)
print(f"Kích thước ảnh cạnh: {edges_cv2.shape}")
print(f"Số pixel cạnh: {np.sum(edges_cv2 > 0)}")
print(f"Tỷ lệ pixel cạnh: {np.sum(edges_cv2 > 0) / edges_cv2.size * 100:.2f}%")

### b) Canny với Scikit-image (`skimage.feature.canny`)

In [ ]:
# ============================================================
# 2.1b - Canny Edge Detection với Scikit-image
# ============================================================

# Chuyển ảnh sang float [0, 1] cho skimage
img_float = img_gray.astype(np.float64) / 255.0

# Áp dụng Canny với skimage
# skimage.feature.canny tích hợp sẵn Gaussian blur
# Tham số: sigma (cho Gaussian), low_threshold, high_threshold
edges_skimage = skimage_canny(
    img_float,
    sigma=1.4,            # Độ lệch chuẩn của Gaussian filter
    low_threshold=0.05,   # Ngưỡng thấp (giá trị float 0-1)
    high_threshold=0.15   # Ngưỡng cao (giá trị float 0-1)
)

# Hiển thị kết quả
show_images(
    [img_gray, edges_skimage.astype(np.uint8) * 255],
    ['Ảnh gốc (Grayscale)',
     'Canny Skimage\n(σ=1.4, T_low=0.05, T_high=0.15)'],
    figsize=(12, 5)
)

print("=" * 60)
print("KẾT QUẢ CANNY VỚI SCIKIT-IMAGE")
print("=" * 60)
print(f"Kích thước ảnh cạnh: {edges_skimage.shape}")
print(f"Kiểu dữ liệu: {edges_skimage.dtype} (True/False)")
print(f"Số pixel cạnh: {np.sum(edges_skimage)}")
print(f"Tỷ lệ pixel cạnh: {np.sum(edges_skimage) / edges_skimage.size * 100:.2f}%")

### c) So sánh kết quả giữa OpenCV và Scikit-image

In [ ]:
# ============================================================
# 2.1c - So sánh trực quan OpenCV vs Scikit-image
# ============================================================

# Chuẩn hóa kết quả về cùng dạng để so sánh
edges_cv2_norm = (edges_cv2 > 0).astype(np.uint8) * 255
edges_sk_norm = edges_skimage.astype(np.uint8) * 255

# Tạo ảnh so sánh sự khác biệt
# Xanh lá = chỉ có trong OpenCV, Đỏ = chỉ có trong Skimage, Trắng = cả hai
diff_img = np.zeros((*img_gray.shape, 3), dtype=np.uint8)
diff_img[..., 0] = edges_sk_norm  # Kênh đỏ - Skimage
diff_img[..., 1] = edges_cv2_norm  # Kênh xanh lá - OpenCV
diff_img[..., 2] = 0

show_images(
    [img_gray, edges_cv2_norm, edges_sk_norm, diff_img],
    ['Ảnh gốc',
     'OpenCV Canny',
     'Scikit-image Canny',
     'So sánh\n(Xanh=OpenCV, Đỏ=Skimage, Vàng=Cả hai)'],
    figsize=(18, 5)
)

# Thống kê so sánh
cv2_pixels = np.sum(edges_cv2 > 0)
sk_pixels = np.sum(edges_skimage)
common_pixels = np.sum((edges_cv2 > 0) & edges_skimage)

print("=" * 60)
print("SO SÁNH OPENCV vs SCIKIT-IMAGE")
print("=" * 60)
print(f"Số pixel cạnh OpenCV:      {cv2_pixels}")
print(f"Số pixel cạnh Scikit-image: {sk_pixels}")
print(f"Pixel cạnh chung:           {common_pixels}")
print(f"Tỷ lệ trùng khớp:          {common_pixels / max(cv2_pixels, sk_pixels) * 100:.1f}%")
print("\n📌 Nhận xét: Sự khác biệt do cách triển khai nội bộ")
print("   (cách tính gradient, cách xử lý ngưỡng, biên ảnh...) của mỗi thư viện.")

## 2.2. Thay đổi các tham số và quan sát kết quả

### a) Ảnh hưởng của Sigma ($\sigma$)

In [ ]:
# ============================================================
# 2.2a - Khảo sát ảnh hưởng của tham số Sigma
# ============================================================

sigma_values = [0.5, 1.0, 1.5, 2.0, 3.0, 5.0]

fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.flatten()

edge_counts = []

for i, sigma in enumerate(sigma_values):
    # Skimage Canny - sigma được tích hợp trực tiếp
    edges = skimage_canny(img_float, sigma=sigma,
                          low_threshold=0.05, high_threshold=0.15)
    count = np.sum(edges)
    edge_counts.append(count)
    
    axes[i].imshow(edges, cmap='gray')
    axes[i].set_title(f'σ = {sigma}\n(Pixels cạnh: {count})', fontweight='bold')
    axes[i].axis('off')

plt.suptitle('ẢNH HƯỞNG CỦA SIGMA (σ) TRONG GAUSSIAN FILTER',
             fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

# Biểu đồ thống kê
fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar([str(s) for s in sigma_values], edge_counts,
              color=plt.cm.viridis(np.linspace(0.2, 0.8, len(sigma_values))),
              edgecolor='black', linewidth=0.5)
ax.set_xlabel('Sigma (σ)', fontweight='bold')
ax.set_ylabel('Số pixel cạnh', fontweight='bold')
ax.set_title('Số lượng pixel cạnh theo giá trị Sigma', fontweight='bold')
for bar, count in zip(bars, edge_counts):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 100,
            str(count), ha='center', fontweight='bold')
plt.tight_layout()
plt.show()

print("📌 NHẬN XÉT:")
print("  • σ nhỏ (0.5): Phát hiện nhiều chi tiết nhưng cũng nhiều nhiễu.")
print("  • σ vừa (1.0-1.5): Cân bằng tốt giữa chi tiết và nhiễu.")
print("  • σ lớn (3.0-5.0): Cạnh mịn hơn, ít chi tiết, mất các cạnh yếu.")

### b) Ảnh hưởng của ngưỡng thấp và ngưỡng cao

In [ ]:
# ============================================================
# 2.2b - Khảo sát ảnh hưởng của ngưỡng thấp và ngưỡng cao
# ============================================================

# --- Phần 1: Thay đổi ngưỡng với OpenCV ---
threshold_pairs = [
    (10, 50),    # Ngưỡng rất thấp
    (30, 100),   # Ngưỡng thấp
    (50, 150),   # Ngưỡng mặc định (tỷ lệ 1:3)
    (100, 200),  # Ngưỡng cao
    (150, 250),  # Ngưỡng rất cao
    (50, 100),   # Tỷ lệ 1:2
]

fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.flatten()

for i, (t_low, t_high) in enumerate(threshold_pairs):
    edges = cv2.Canny(img_blur, t_low, t_high)
    count = np.sum(edges > 0)
    ratio = t_high / t_low
    
    axes[i].imshow(edges, cmap='gray')
    axes[i].set_title(
        f'T_low={t_low}, T_high={t_high}\n'
        f'Tỷ lệ={ratio:.1f}:1 | Pixels={count}',
        fontweight='bold'
    )
    axes[i].axis('off')

plt.suptitle('ẢNH HƯỞNG CỦA NGƯỠNG THẤP VÀ NGƯỠNG CAO (OpenCV)',
             fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

# --- Phần 2: Thay đổi ngưỡng với Scikit-image ---
threshold_pairs_sk = [
    (0.01, 0.05),   # Ngưỡng rất thấp
    (0.03, 0.10),   # Ngưỡng thấp
    (0.05, 0.15),   # Mặc định
    (0.10, 0.20),   # Ngưỡng cao
    (0.15, 0.30),   # Ngưỡng rất cao
    (0.05, 0.10),   # Tỷ lệ 1:2
]

fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.flatten()

for i, (t_low, t_high) in enumerate(threshold_pairs_sk):
    edges = skimage_canny(img_float, sigma=1.4,
                          low_threshold=t_low, high_threshold=t_high)
    count = np.sum(edges)
    ratio = t_high / t_low
    
    axes[i].imshow(edges, cmap='gray')
    axes[i].set_title(
        f'T_low={t_low}, T_high={t_high}\n'
        f'Tỷ lệ={ratio:.1f}:1 | Pixels={count}',
        fontweight='bold'
    )
    axes[i].axis('off')

plt.suptitle('ẢNH HƯỞNG CỦA NGƯỠNG THẤP VÀ NGƯỠNG CAO (Scikit-image)',
             fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("📌 NHẬN XÉT:")
print("  • Ngưỡng thấp → phát hiện nhiều cạnh (bao gồm cả nhiễu).")
print("  • Ngưỡng cao → chỉ giữ cạnh mạnh, có thể bỏ sót cạnh quan trọng.")
print("  • Tỷ lệ T_high/T_low ≈ 2:1 đến 3:1 thường cho kết quả tốt nhất.")

### c) So sánh Canny với các phương pháp khác (Sobel, Laplacian)

In [ ]:
# ============================================================
# 2.2c - So sánh Canny với Sobel và Laplacian
# ============================================================

# 1. Sobel Edge Detection
sobel_x = cv2.Sobel(img_blur, cv2.CV_64F, 1, 0, ksize=3)
sobel_y = cv2.Sobel(img_blur, cv2.CV_64F, 0, 1, ksize=3)
sobel_mag = np.sqrt(sobel_x**2 + sobel_y**2)
sobel_mag = np.uint8(sobel_mag / sobel_mag.max() * 255)

# 2. Laplacian Edge Detection
laplacian = cv2.Laplacian(img_blur, cv2.CV_64F)
laplacian = np.uint8(np.abs(laplacian) / np.abs(laplacian).max() * 255)

# 3. Canny
canny_result = cv2.Canny(img_blur, 50, 150)

# Hiển thị so sánh
show_images(
    [img_gray, sobel_mag, laplacian, canny_result],
    ['Ảnh gốc',
     'Sobel\n(Gradient magnitude)',
     'Laplacian\n(Đạo hàm bậc 2)',
     'Canny\n(T_low=50, T_high=150)'],
    figsize=(18, 5)
)

print("📌 SO SÁNH TRỰC QUAN:")
print("  • Sobel: Cho cạnh dày, hiển thị gradient magnitude, không có NMS.")
print("  • Laplacian: Nhạy nhiễu, cạnh kép, khó xác định biên rõ ràng.")
print("  • Canny: Cạnh mỏng, sắc nét, ít nhiễu, liên tục → chất lượng cao nhất.")

## 2.3. Áp dụng Canny cho các loại ảnh khác nhau

### a) Ảnh có nhiều nhiễu

In [ ]:
# ============================================================
# 2.3a - Canny trên ảnh có nhiều nhiễu
# ============================================================

# Tạo ảnh nhiễu từ ảnh gốc
np.random.seed(42)
noise_levels = [0.02, 0.05, 0.1]  # Mức nhiễu Gaussian

fig, axes = plt.subplots(3, 4, figsize=(20, 15))

for row, noise_level in enumerate(noise_levels):
    # Thêm nhiễu Gaussian
    noisy_img = util.random_noise(img_float, mode='gaussian', var=noise_level)
    noisy_uint8 = np.uint8(np.clip(noisy_img, 0, 1) * 255)
    
    # Canny với sigma nhỏ (ít làm mịn)
    edges_low_sigma = skimage_canny(noisy_img, sigma=1.0,
                                     low_threshold=0.05, high_threshold=0.15)
    
    # Canny với sigma mặc định
    edges_mid_sigma = skimage_canny(noisy_img, sigma=2.0,
                                     low_threshold=0.05, high_threshold=0.15)
    
    # Canny với sigma lớn (làm mịn mạnh)
    edges_high_sigma = skimage_canny(noisy_img, sigma=3.0,
                                      low_threshold=0.05, high_threshold=0.15)
    
    axes[row, 0].imshow(noisy_uint8, cmap='gray')
    axes[row, 0].set_title(f'Ảnh nhiễu\n(var={noise_level})', fontweight='bold')
    
    axes[row, 1].imshow(edges_low_sigma, cmap='gray')
    axes[row, 1].set_title(f'σ=1.0\n({np.sum(edges_low_sigma)} pixels)', fontweight='bold')
    
    axes[row, 2].imshow(edges_mid_sigma, cmap='gray')
    axes[row, 2].set_title(f'σ=2.0\n({np.sum(edges_mid_sigma)} pixels)', fontweight='bold')
    
    axes[row, 3].imshow(edges_high_sigma, cmap='gray')
    axes[row, 3].set_title(f'σ=3.0\n({np.sum(edges_high_sigma)} pixels)', fontweight='bold')
    
    for j in range(4):
        axes[row, j].axis('off')

plt.suptitle('CANNY TRÊN ẢNH CÓ NHIỄU VỚI CÁC MỨC SIGMA KHÁC NHAU',
             fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("📌 KẾT LUẬN:")
print("  • Ảnh nhiễu nhiều cần sigma lớn hơn để loại bỏ nhiễu.")
print("  • σ=1.0 trên ảnh nhiễu → phát hiện quá nhiều cạnh giả.")
print("  • σ=2.0-3.0 giúp loại bỏ nhiễu tốt nhưng có thể mất chi tiết.")

### b) Ảnh có tương phản thấp

In [ ]:
# ============================================================
# 2.3b - Canny trên ảnh tương phản thấp
# ============================================================

# Tạo ảnh tương phản thấp bằng cách nén dải giá trị
img_low_contrast = np.uint8(img_gray * 0.3 + 90)  # Nén về [90, 166]
img_low_float = img_low_contrast.astype(np.float64) / 255.0

# Cải thiện tương phản bằng CLAHE (Contrast Limited Adaptive Histogram Equalization)
clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8, 8))
img_enhanced = clahe.apply(img_low_contrast)
img_enhanced_float = img_enhanced.astype(np.float64) / 255.0

# Áp dụng Canny
edges_low = skimage_canny(img_low_float, sigma=1.4,
                           low_threshold=0.03, high_threshold=0.10)
edges_enhanced = skimage_canny(img_enhanced_float, sigma=1.4,
                                low_threshold=0.05, high_threshold=0.15)
edges_original = skimage_canny(img_float, sigma=1.4,
                                low_threshold=0.05, high_threshold=0.15)

fig, axes = plt.subplots(2, 3, figsize=(18, 12))

# Hàng 1: Ảnh gốc
axes[0, 0].imshow(img_gray, cmap='gray')
axes[0, 0].set_title('Ảnh gốc', fontweight='bold')
axes[0, 1].imshow(img_low_contrast, cmap='gray')
axes[0, 1].set_title(f'Ảnh tương phản thấp\n(min={img_low_contrast.min()}, max={img_low_contrast.max()})',
                     fontweight='bold')
axes[0, 2].imshow(img_enhanced, cmap='gray')
axes[0, 2].set_title('Sau CLAHE\n(tăng tương phản)', fontweight='bold')

# Hàng 2: Kết quả Canny
axes[1, 0].imshow(edges_original, cmap='gray')
axes[1, 0].set_title(f'Canny ảnh gốc\n({np.sum(edges_original)} pixels)', fontweight='bold')
axes[1, 1].imshow(edges_low, cmap='gray')
axes[1, 1].set_title(f'Canny tương phản thấp\n({np.sum(edges_low)} pixels)', fontweight='bold')
axes[1, 2].imshow(edges_enhanced, cmap='gray')
axes[1, 2].set_title(f'Canny sau CLAHE\n({np.sum(edges_enhanced)} pixels)', fontweight='bold')

for ax in axes.flatten():
    ax.axis('off')

plt.suptitle('CANNY TRÊN ẢNH TƯƠNG PHẢN THẤP', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("📌 KẾT LUẬN:")
print("  • Ảnh tương phản thấp → Canny phát hiện ít cạnh hơn.")
print("  • Giải pháp: Tiền xử lý bằng CLAHE hoặc histogram equalization.")
print("  • Hoặc giảm ngưỡng (T_low, T_high) để phát hiện cạnh yếu.")

### c) Ảnh có nhiều chi tiết

In [ ]:
# ============================================================
# 2.3c - Canny trên ảnh có nhiều chi tiết
# ============================================================

# Sử dụng ảnh coins (nhiều chi tiết hình tròn) và ảnh brick
img_coins = data.coins()  # Ảnh đồng xu - nhiều chi tiết hình tròn
img_text = data.page()    # Ảnh trang sách - nhiều chi tiết text

# Chuẩn hóa sang float
coins_float = img_coins.astype(np.float64) / 255.0
text_float = img_text.astype(np.float64) / 255.0

# Áp dụng Canny với các sigma khác nhau
sigmas = [0.5, 1.5, 3.0]

fig, axes = plt.subplots(2, 4, figsize=(20, 10))

# Hàng 1: Ảnh coins
axes[0, 0].imshow(img_coins, cmap='gray')
axes[0, 0].set_title('Coins (gốc)', fontweight='bold')
for i, sigma in enumerate(sigmas):
    edges = skimage_canny(coins_float, sigma=sigma,
                          low_threshold=0.04, high_threshold=0.12)
    axes[0, i+1].imshow(edges, cmap='gray')
    axes[0, i+1].set_title(f'σ={sigma} ({np.sum(edges)} px)', fontweight='bold')

# Hàng 2: Ảnh text
axes[1, 0].imshow(img_text, cmap='gray')
axes[1, 0].set_title('Page (gốc)', fontweight='bold')
for i, sigma in enumerate(sigmas):
    edges = skimage_canny(text_float, sigma=sigma,
                          low_threshold=0.04, high_threshold=0.12)
    axes[1, i+1].imshow(edges, cmap='gray')
    axes[1, i+1].set_title(f'σ={sigma} ({np.sum(edges)} px)', fontweight='bold')

for ax in axes.flatten():
    ax.axis('off')

plt.suptitle('CANNY TRÊN ẢNH CÓ NHIỀU CHI TIẾT', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("📌 KẾT LUẬN:")
print("  • Ảnh nhiều chi tiết → cần σ phù hợp:")
print("    - σ nhỏ: Giữ tất cả chi tiết, kết quả dày đặc.")
print("    - σ lớn: Chỉ giữ cạnh chính, mất chi tiết nhỏ.")
print("  • Tuỳ mục đích: cần chi tiết → σ nhỏ; cần tổng quan → σ lớn.")

### d) Tổng hợp và đánh giá

In [ ]:
# ============================================================
# 2.3d - Bảng tổng hợp đánh giá
# ============================================================

print("=" * 80)
print("BẢNG TỔNG HỢP: THAM SỐ CANNY PHÙ HỢP CHO TỪNG LOẠI ẢNH")
print("=" * 80)
print(f"{'Loại ảnh':<25} {'Sigma (σ)':<15} {'T_low':<12} {'T_high':<12} {'Ghi chú'}")
print("-" * 80)
print(f"{'Ảnh bình thường':<25} {'1.0 - 1.5':<15} {'0.05':<12} {'0.15':<12} {'Tham số mặc định'}")
print(f"{'Ảnh nhiều nhiễu':<25} {'2.0 - 4.0':<15} {'0.05':<12} {'0.15':<12} {'Tăng σ để lọc nhiễu'}")
print(f"{'Ảnh tương phản thấp':<25} {'1.0 - 1.5':<15} {'0.02':<12} {'0.08':<12} {'Giảm ngưỡng hoặc CLAHE'}")
print(f"{'Ảnh nhiều chi tiết':<25} {'0.5 - 1.0':<15} {'0.03':<12} {'0.10':<12} {'Giảm σ để giữ chi tiết'}")
print("=" * 80)
print("\n📌 KẾT LUẬN CHUNG:")
print("  Không có bộ tham số tối ưu cho mọi ảnh.")
print("  Cần điều chỉnh σ, T_low, T_high phù hợp với đặc điểm của từng ảnh.")
print("  Tiền xử lý (lọc nhiễu, tăng tương phản) giúp cải thiện kết quả đáng kể.")

## 2.4. Kết hợp Canny với các kỹ thuật khác

### a) Kết hợp Canny với phân đoạn ảnh (Segmentation) – Tìm vùng

In [ ]:
# ============================================================
# 2.4a - Kết hợp Canny + Phân đoạn ảnh (tìm contours/vùng)
# ============================================================

# Sử dụng ảnh coins
img_coins = data.coins()

# Bước 1: Áp dụng Gaussian Blur
coins_blur = cv2.GaussianBlur(img_coins, (5, 5), 1.5)

# Bước 2: Phát hiện cạnh bằng Canny
edges_coins = cv2.Canny(coins_blur, 30, 100)

# Bước 3: Giãn nở (dilate) để nối các cạnh gần nhau
kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
edges_dilated = cv2.dilate(edges_coins, kernel, iterations=2)

# Bước 4: Lấp đầy lỗ trống bên trong contours
edges_filled = ndimage.binary_fill_holes(edges_dilated).astype(np.uint8) * 255

# Bước 5: Tìm contours
contours, hierarchy = cv2.findContours(
    edges_filled, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE
)

# Bước 6: Lọc contours theo diện tích (loại bỏ nhiễu nhỏ)
min_area = 500
filtered_contours = [c for c in contours if cv2.contourArea(c) > min_area]

# Bước 7: Vẽ kết quả
result_segmentation = cv2.cvtColor(img_coins, cv2.COLOR_GRAY2BGR)
cv2.drawContours(result_segmentation, filtered_contours, -1, (0, 255, 0), 2)

# Tạo mask phân đoạn
mask = np.zeros_like(img_coins)
cv2.drawContours(mask, filtered_contours, -1, 255, -1)  # Tô đầy

# Hiển thị các bước
fig, axes = plt.subplots(2, 3, figsize=(18, 12))

axes[0, 0].imshow(img_coins, cmap='gray')
axes[0, 0].set_title('1. Ảnh gốc (Coins)', fontweight='bold')

axes[0, 1].imshow(edges_coins, cmap='gray')
axes[0, 1].set_title('2. Canny Edges', fontweight='bold')

axes[0, 2].imshow(edges_dilated, cmap='gray')
axes[0, 2].set_title('3. Sau Dilate', fontweight='bold')

axes[1, 0].imshow(edges_filled, cmap='gray')
axes[1, 0].set_title('4. Lấp đầy lỗ trống', fontweight='bold')

axes[1, 1].imshow(mask, cmap='gray')
axes[1, 1].set_title(f'5. Mask ({len(filtered_contours)} vùng)', fontweight='bold')

axes[1, 2].imshow(cv2.cvtColor(result_segmentation, cv2.COLOR_BGR2RGB))
axes[1, 2].set_title(f'6. Kết quả phân đoạn\n({len(filtered_contours)} đối tượng)', fontweight='bold')

for ax in axes.flatten():
    ax.axis('off')

plt.suptitle('KẾT HỢP CANNY + PHÂN ĐOẠN ẢNH (TÌM VÙNG)', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print(f"📌 Phát hiện được {len(filtered_contours)} vùng/đối tượng (contours có diện tích > {min_area} px).")

### b) Kết hợp Canny với nhận dạng hình dạng (Shape Detection)

In [ ]:
# ============================================================
# 2.4b - Kết hợp Canny + Nhận dạng hình dạng
# ============================================================

# --- Phương pháp 1: Hough Circle Transform ---
# Phát hiện hình tròn trong ảnh coins

img_coins = data.coins()
coins_blur = cv2.GaussianBlur(img_coins, (9, 9), 2)

# Phát hiện đường tròn bằng Hough Circle Transform
circles = cv2.HoughCircles(
    coins_blur,
    cv2.HOUGH_GRADIENT,
    dp=1,             # Tỷ lệ nghịch đảo độ phân giải accumulator
    minDist=50,       # Khoảng cách tối thiểu giữa các tâm tròn
    param1=100,       # Ngưỡng cao cho Canny (bên trong HoughCircles)
    param2=30,        # Ngưỡng accumulator
    minRadius=20,     # Bán kính tối thiểu
    maxRadius=60      # Bán kính tối đa
)

# Vẽ kết quả
result_circles = cv2.cvtColor(img_coins, cv2.COLOR_GRAY2BGR)
if circles is not None:
    circles = np.uint16(np.around(circles))
    for (x, y, r) in circles[0, :]:
        cv2.circle(result_circles, (x, y), r, (0, 255, 0), 2)    # Vòng tròn
        cv2.circle(result_circles, (x, y), 3, (0, 0, 255), -1)   # Tâm

# --- Phương pháp 2: Nhận dạng đa giác bằng approxPolyDP ---

# Tạo ảnh có nhiều hình dạng khác nhau
shape_img = np.zeros((400, 600), dtype=np.uint8)

# Vẽ các hình
cv2.rectangle(shape_img, (50, 50), (180, 180), 255, 3)           # Hình vuông
cv2.rectangle(shape_img, (220, 50), (400, 150), 255, 3)          # Hình chữ nhật
cv2.circle(shape_img, (500, 100), 60, 255, 3)                    # Hình tròn
pts_tri = np.array([[100, 350], [50, 250], [150, 250]], np.int32)
cv2.polylines(shape_img, [pts_tri], True, 255, 3)                # Tam giác
pts_pent = np.array([[300, 230], [250, 280], [270, 340],
                      [330, 340], [350, 280]], np.int32)
cv2.polylines(shape_img, [pts_pent], True, 255, 3)               # Ngũ giác
cv2.ellipse(shape_img, (500, 300), (70, 40), 0, 0, 360, 255, 3) # Elip

# Canny trên ảnh hình dạng
edges_shapes = cv2.Canny(shape_img, 50, 150)

# Tìm và nhận dạng contours
contours_shapes, _ = cv2.findContours(
    edges_shapes, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE
)

result_shapes = cv2.cvtColor(shape_img, cv2.COLOR_GRAY2BGR)
shape_names = []

for contour in contours_shapes:
    # Xấp xỉ đa giác
    epsilon = 0.03 * cv2.arcLength(contour, True)
    approx = cv2.approxPolyDP(contour, epsilon, True)
    n_vertices = len(approx)
    
    # Xác định hình dạng dựa trên số đỉnh
    if n_vertices == 3:
        shape_name = "Tam giac"
        color = (0, 255, 255)  # Vàng
    elif n_vertices == 4:
        # Phân biệt hình vuông và hình chữ nhật
        (x, y, w, h) = cv2.boundingRect(approx)
        ratio = w / float(h)
        shape_name = "Hinh vuong" if 0.9 <= ratio <= 1.1 else "HCN"
        color = (0, 255, 0)   # Xanh lá
    elif n_vertices == 5:
        shape_name = "Ngu giac"
        color = (255, 0, 0)   # Xanh dương
    elif n_vertices > 6:
        # Kiểm tra circularity để phân biệt tròn/elip
        area = cv2.contourArea(contour)
        perimeter = cv2.arcLength(contour, True)
        circularity = 4 * np.pi * area / (perimeter ** 2) if perimeter > 0 else 0
        shape_name = "Hinh tron" if circularity > 0.85 else "Elip"
        color = (0, 0, 255)   # Đỏ
    else:
        shape_name = f"{n_vertices} canh"
        color = (255, 255, 0)
    
    shape_names.append(shape_name)
    
    # Vẽ contour và nhãn
    cv2.drawContours(result_shapes, [contour], -1, color, 2)
    M = cv2.moments(contour)
    if M["m00"] != 0:
        cx = int(M["m10"] / M["m00"])
        cy = int(M["m01"] / M["m00"])
        cv2.putText(result_shapes, shape_name, (cx - 30, cy),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)

# Hiển thị kết quả
fig, axes = plt.subplots(1, 4, figsize=(22, 6))

axes[0].imshow(img_coins, cmap='gray')
axes[0].set_title('Ảnh Coins (gốc)', fontweight='bold')

axes[1].imshow(cv2.cvtColor(result_circles, cv2.COLOR_BGR2RGB))
n_circles = len(circles[0]) if circles is not None else 0
axes[1].set_title(f'Hough Circle\n({n_circles} hình tròn)', fontweight='bold')

axes[2].imshow(shape_img, cmap='gray')
axes[2].set_title('Ảnh hình dạng (gốc)', fontweight='bold')

axes[3].imshow(cv2.cvtColor(result_shapes, cv2.COLOR_BGR2RGB))
axes[3].set_title(f'Nhận dạng hình dạng\n({len(shape_names)} đối tượng)', fontweight='bold')

for ax in axes:
    ax.axis('off')

plt.suptitle('KẾT HỢP CANNY + NHẬN DẠNG HÌNH DẠNG', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print(f"📌 Hough Circle: Phát hiện {n_circles} hình tròn trong ảnh coins.")
print(f"📌 Shape Detection: Nhận dạng {len(shape_names)} hình dạng: {', '.join(shape_names)}")

### c) Kết hợp Canny + Hough Line Transform (phát hiện đường thẳng)

In [ ]:
# ============================================================
# 2.4c - Kết hợp Canny + Hough Line Transform
# ============================================================

# Sử dụng ảnh checkerboard để phát hiện đường thẳng
img_checker = data.checkerboard()
checker_blur = cv2.GaussianBlur(img_checker, (5, 5), 1.5)

# Phát hiện cạnh bằng Canny
edges_checker = cv2.Canny(checker_blur, 50, 150)

# Phát hiện đường thẳng bằng Hough Line Transform
lines = cv2.HoughLinesP(
    edges_checker,
    rho=1,              # Độ phân giải khoảng cách (pixel)
    theta=np.pi / 180,  # Độ phân giải góc (radian)
    threshold=50,       # Ngưỡng accumulator
    minLineLength=30,   # Chiều dài tối thiểu
    maxLineGap=10       # Khoảng cách tối đa giữa các điểm
)

# Vẽ đường thẳng lên ảnh
result_lines = cv2.cvtColor(img_checker, cv2.COLOR_GRAY2BGR)
if lines is not None:
    for line in lines:
        x1, y1, x2, y2 = line[0]
        cv2.line(result_lines, (x1, y1), (x2, y2), (0, 0, 255), 2)

# Hiển thị
show_images(
    [img_checker, edges_checker,
     cv2.cvtColor(result_lines, cv2.COLOR_BGR2RGB)],
    ['Ảnh Checkerboard (gốc)',
     'Canny Edges',
     f'Hough Lines\n({len(lines) if lines is not None else 0} đường thẳng)'],
    figsize=(16, 5)
)

print(f"📌 Phát hiện {len(lines) if lines is not None else 0} đường thẳng.")
print("📌 Ứng dụng: Phát hiện làn đường, cạnh toà nhà, kiểm tra góc vuông sản phẩm...")

---
# III. CÁC CÂU HỎI MỞ RỘNG

## 3.1. Làm thế nào để đánh giá chất lượng của các cạnh được phát hiện bởi Canny?

Có nhiều phương pháp đánh giá chất lượng phát hiện cạnh:

### Phương pháp định lượng (Quantitative)

1. **So sánh với Ground Truth (cạnh chuẩn):**
   - Sử dụng các bộ dữ liệu có nhãn cạnh như **BSDS500 (Berkeley Segmentation Dataset)**.
   - Tính các chỉ số:
     - **Precision** = TP / (TP + FP): Tỷ lệ pixel cạnh đúng trong tổng pixel cạnh phát hiện.
     - **Recall** = TP / (TP + FN): Tỷ lệ pixel cạnh đúng được phát hiện.
     - **F1-score** = 2 × (Precision × Recall) / (Precision + Recall).

2. **Pratt's Figure of Merit (FOM):**
   $$FOM = \frac{1}{\max(N_{ideal}, N_{detected})} \sum_{i=1}^{N_{detected}} \frac{1}{1 + \alpha \cdot d_i^2}$$
   - Trong đó $d_i$ là khoảng cách từ pixel cạnh phát hiện đến pixel cạnh lý tưởng gần nhất.
   - FOM ∈ [0, 1], giá trị càng gần 1 càng tốt.

3. **Peak Signal-to-Noise Ratio (PSNR):** So sánh ảnh cạnh phát hiện với ground truth.

### Phương pháp định tính (Qualitative)

4. **Đánh giá trực quan:**
   - Cạnh có liên tục không? (Continuity)
   - Cạnh có mỏng không? (Thinness - 1 pixel)
   - Có ít cạnh giả (false edges) không?
   - Có bỏ sót cạnh quan trọng không?

5. **Ba tiêu chí tối ưu của Canny:**
   - **Good detection**: Xác suất bỏ sót cạnh thật thấp, xác suất phát hiện cạnh giả thấp.
   - **Good localization**: Cạnh phát hiện sát với vị trí cạnh thật.
   - **Single response**: Mỗi cạnh thật chỉ được phát hiện một lần.

## 3.2. Có những phương pháp nào khác để cải thiện hiệu suất của Canny?

### Cải thiện ở bước tiền xử lý

1. **Sử dụng bộ lọc thay thế Gaussian:**
   - **Bilateral Filter**: Giảm nhiễu nhưng giữ cạnh sắc nét hơn Gaussian.
   - **Non-Local Means Denoising**: Hiệu quả cao cho ảnh nhiễu phức tạp.
   - **Anisotropic Diffusion**: Làm mịn thích ứng theo hướng gradient.

2. **Tăng cường tương phản:**
   - CLAHE (Contrast Limited Adaptive Histogram Equalization).
   - Histogram Equalization.
   - Gamma correction.

### Cải thiện ở bước xử lý chính

3. **Tự động chọn ngưỡng (Automatic Thresholding):**
   - **Phương pháp Otsu**: Tự động tìm ngưỡng tối ưu dựa trên histogram của gradient.
   - **Phương pháp Median**: $T_{high} = 0.33 \times \text{median}(I)$, $T_{low} = 0.66 \times \text{median}(I)$.
   - **Adaptive Canny**: Thay đổi ngưỡng cục bộ theo vùng ảnh.

4. **Sử dụng gradient tốt hơn:**
   - Thay Sobel bằng **Scharr operator** (chính xác hơn cho gradient).
   - Sử dụng **Deriche filter** (IIR filter, hiệu quả hơn cho ảnh lớn).

### Cải thiện ở bước hậu xử lý

5. **Morphological operations:**
   - Dilation + Erosion để nối các cạnh đứt đoạn.
   - Skeletonization để làm mỏng cạnh hơn.

6. **Machine Learning / Deep Learning:**
   - **Structured Edge Detection (Dollar, 2013)**: Sử dụng Random Forest.
   - **HED (Holistically-Nested Edge Detection)**: Sử dụng Deep Learning, cho kết quả tốt hơn Canny trên nhiều benchmark.
   - **RCF (Richer Convolutional Features)**: Cải tiến từ HED.

## 3.3. Canny có thể được sử dụng để phát hiện cạnh trong ảnh màu không?

**Có**, Canny có thể áp dụng cho ảnh màu. Có nhiều cách tiếp cận:

### Cách 1: Chuyển sang ảnh xám (phổ biến nhất)
- Chuyển ảnh RGB → Grayscale, rồi áp dụng Canny bình thường.
- **Ưu điểm**: Đơn giản, nhanh.
- **Nhược điểm**: Mất thông tin màu sắc → bỏ sót cạnh giữa các vùng có cùng độ sáng nhưng khác màu.

### Cách 2: Áp dụng Canny trên từng kênh màu riêng biệt
- Tách ảnh thành 3 kênh (R, G, B), áp dụng Canny cho mỗi kênh.
- Kết hợp kết quả bằng phép OR (hợp) hoặc AND (giao).
- **Ưu điểm**: Phát hiện được cạnh ở tất cả các kênh.
- **Nhược điểm**: Có thể tạo ra nhiều cạnh giả.

### Cách 3: Sử dụng không gian màu khác
- Chuyển sang HSV, Lab, hoặc YCrCb.
- Áp dụng Canny trên kênh phù hợp (ví dụ: kênh L trong Lab cho độ sáng).
- **Ưu điểm**: Tận dụng tốt hơn thông tin màu sắc.

### Cách 4: Color Gradient (nâng cao)
- Tính gradient đa biến (multi-channel gradient) bằng cách sử dụng **Di Zenzo gradient**.
- Tính ma trận gradient tensor cho ảnh RGB, lấy eigenvalue lớn nhất làm cường độ gradient.
- **Ưu điểm**: Chính xác nhất cho ảnh màu.
- **Nhược điểm**: Phức tạp hơn, tốn tính toán.

In [ ]:
# ============================================================
# Minh họa: Canny trên ảnh màu với các cách tiếp cận khác nhau
# ============================================================

# Tải ảnh màu từ skimage
img_color = data.astronaut()  # Ảnh phi hành gia (RGB)

# --- Cách 1: Chuyển sang grayscale ---
gray = cv2.cvtColor(img_color, cv2.COLOR_RGB2GRAY)
edges_gray = cv2.Canny(cv2.GaussianBlur(gray, (5, 5), 1.4), 50, 150)

# --- Cách 2: Canny trên từng kênh RGB ---
edges_r = cv2.Canny(cv2.GaussianBlur(img_color[:,:,0], (5,5), 1.4), 50, 150)
edges_g = cv2.Canny(cv2.GaussianBlur(img_color[:,:,1], (5,5), 1.4), 50, 150)
edges_b = cv2.Canny(cv2.GaussianBlur(img_color[:,:,2], (5,5), 1.4), 50, 150)
# Kết hợp bằng phép OR
edges_rgb_combined = np.maximum(np.maximum(edges_r, edges_g), edges_b)

# --- Cách 3: Sử dụng không gian màu Lab ---
img_lab = cv2.cvtColor(img_color, cv2.COLOR_RGB2Lab)
edges_L = cv2.Canny(cv2.GaussianBlur(img_lab[:,:,0], (5,5), 1.4), 50, 150)
edges_a = cv2.Canny(cv2.GaussianBlur(img_lab[:,:,1], (5,5), 1.4), 50, 150)
edges_lab = np.maximum(edges_L, edges_a)

# Hiển thị kết quả
fig, axes = plt.subplots(2, 3, figsize=(18, 12))

axes[0, 0].imshow(img_color)
axes[0, 0].set_title('Ảnh gốc (RGB)', fontweight='bold')

axes[0, 1].imshow(gray, cmap='gray')
axes[0, 1].set_title('Chuyển Grayscale', fontweight='bold')

axes[0, 2].imshow(edges_gray, cmap='gray')
axes[0, 2].set_title(f'Cách 1: Canny trên Gray\n({np.sum(edges_gray>0)} px)', fontweight='bold')

axes[1, 0].imshow(edges_rgb_combined, cmap='gray')
axes[1, 0].set_title(f'Cách 2: Canny RGB (OR)\n({np.sum(edges_rgb_combined>0)} px)', fontweight='bold')

axes[1, 1].imshow(edges_lab, cmap='gray')
axes[1, 1].set_title(f'Cách 3: Canny Lab (L+a)\n({np.sum(edges_lab>0)} px)', fontweight='bold')

# So sánh: overlay cạnh lên ảnh gốc
overlay = img_color.copy()
overlay[edges_rgb_combined > 0] = [255, 0, 0]  # Tô đỏ lên cạnh
axes[1, 2].imshow(overlay)
axes[1, 2].set_title('Overlay cạnh lên ảnh gốc', fontweight='bold')

for ax in axes.flatten():
    ax.axis('off')

plt.suptitle('CANNY TRÊN ẢNH MÀU - CÁC CÁCH TIẾP CẬN', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("📌 NHẬN XÉT:")
print("  • Cách 2 (RGB combined) phát hiện nhiều cạnh nhất nhờ thông tin đa kênh.")
print("  • Cách 1 (Grayscale) đơn giản, nhanh, phù hợp cho hầu hết trường hợp.")
print("  • Cách 3 (Lab) tách biệt tốt hơn giữa độ sáng và màu sắc.")

## 3.4. Làm thế nào để áp dụng Canny cho các video?

### Nguyên tắc cơ bản

Áp dụng Canny cho video chính là **áp dụng Canny cho từng frame** của video. Quy trình:

1. **Đọc video** bằng `cv2.VideoCapture()`.
2. **Lặp qua từng frame:**
   - Chuyển frame sang grayscale.
   - Áp dụng Gaussian Blur.
   - Áp dụng Canny.
3. **Hiển thị hoặc lưu kết quả.**

### Các kỹ thuật tối ưu cho video

- **Temporal smoothing**: Lấy trung bình cạnh qua nhiều frame liên tiếp để giảm flicker (nhấp nháy).
- **Adaptive thresholding**: Thay đổi ngưỡng tự động theo điều kiện ánh sáng của từng frame.
- **Background subtraction + Canny**: Chỉ phát hiện cạnh trên vùng chuyển động (foreground).
- **GPU acceleration**: Sử dụng `cv2.cuda` cho xử lý thời gian thực.
- **Xử lý đa luồng**: Đọc frame và xử lý song song.

### Mã giả (Pseudocode)

```python
cap = cv2.VideoCapture('video.mp4')  # hoặc 0 cho webcam

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break
    
    # Chuyển grayscale
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    
    # Giảm nhiễu
    blur = cv2.GaussianBlur(gray, (5, 5), 1.4)
    
    # Canny
    edges = cv2.Canny(blur, 50, 150)
    
    # Hiển thị
    cv2.imshow('Canny Video', edges)
    
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()
```

### Ứng dụng Canny trên video

- **Lane detection (phát hiện làn đường)** trong xe tự lái.
- **Motion edge detection**: Phát hiện chuyển động qua cạnh thay đổi.
- **Video surveillance**: Giám sát an ninh, phát hiện xâm nhập.
- **Augmented Reality (AR)**: Phát hiện cạnh đối tượng để overlay thông tin.

In [ ]:
# ============================================================
# Minh họa: Mô phỏng Canny trên "video" (chuỗi ảnh)
# ============================================================

# Tạo chuỗi ảnh mô phỏng video (di chuyển vùng sáng)
frames = []
edges_frames = []

for i in range(6):
    # Tạo frame với hình tròn di chuyển
    frame = np.zeros((200, 300), dtype=np.uint8)
    # Hình tròn di chuyển sang phải
    cx = 50 + i * 40
    cv2.circle(frame, (cx, 100), 40, 200, -1)
    cv2.rectangle(frame, (10, 10), (80, 50), 150, -1)
    
    # Thêm nhiễu nhẹ
    noise = np.random.normal(0, 10, frame.shape).astype(np.int16)
    frame_noisy = np.clip(frame.astype(np.int16) + noise, 0, 255).astype(np.uint8)
    
    # Áp dụng Canny
    blur = cv2.GaussianBlur(frame_noisy, (5, 5), 1.0)
    edges = cv2.Canny(blur, 30, 100)
    
    frames.append(frame_noisy)
    edges_frames.append(edges)

# Hiển thị chuỗi frame
fig, axes = plt.subplots(2, 6, figsize=(20, 7))

for i in range(6):
    axes[0, i].imshow(frames[i], cmap='gray')
    axes[0, i].set_title(f'Frame {i+1}', fontweight='bold', fontsize=10)
    axes[0, i].axis('off')
    
    axes[1, i].imshow(edges_frames[i], cmap='gray')
    axes[1, i].set_title(f'Canny F{i+1}', fontweight='bold', fontsize=10)
    axes[1, i].axis('off')

plt.suptitle('MÔ PHỎNG CANNY TRÊN VIDEO (CHUỖI FRAME)',
             fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("📌 Mô phỏng: Hình tròn di chuyển sang phải qua 6 frame.")
print("📌 Canny được áp dụng độc lập cho từng frame.")
print("📌 Trong thực tế, sử dụng cv2.VideoCapture() để đọc video thật.")

---

# TỔNG KẾT

## Những điểm chính đã học:

1. **Thuật toán Canny** gồm 5 bước: Gaussian smoothing → Gradient computation → Non-maximum suppression → Double thresholding → Edge tracking by hysteresis.

2. **Ba tham số quan trọng**: $\sigma$ (mức làm mịn), $T_{low}$ (ngưỡng thấp), $T_{high}$ (ngưỡng cao) — cần được điều chỉnh phù hợp với từng loại ảnh.

3. **So sánh**: Canny cho kết quả **chính xác nhất** so với Sobel và Laplacian nhờ các bước NMS và Hysteresis, nhưng **chậm hơn**.

4. **Với ảnh nhiễu**: Tăng $\sigma$ để lọc nhiễu. **Với ảnh tương phản thấp**: Tiền xử lý bằng CLAHE hoặc giảm ngưỡng.

5. **Kết hợp Canny** với findContours, Hough Transform, approxPolyDP để thực hiện phân đoạn ảnh và nhận dạng hình dạng.

6. Canny có thể áp dụng cho **ảnh màu** (qua nhiều kênh) và **video** (xử lý từng frame).